# Session Duration Pattern Analysis

## Project Overview
Analyze user session data to understand what influences short vs long sessions, device effects, and engagement patterns.

## Learning Objectives
- Analyze session duration distributions and identify right-skewness patterns
- Compare session behavior across devices, traffic sources, and user types
- Identify engagement tiers (bounce, casual, engaged, power user)
- Detect time-of-day and day-of-week effects on session length
- Understand the relationship between session duration and conversion

## Problem Statement
A web analytics team wants to understand session duration patterns to optimize content, identify disengaged users for re-targeting, and improve the onboarding flow for short-session visitors.

## Why This Project Matters
Session duration is a core engagement metric. Short sessions signal UX friction or irrelevant content. Understanding what drives longer sessions helps product teams prioritize features and content that keep users engaged.

## Dataset Overview
Synthetic web session data: ~12,000 sessions over 2 months with duration, pages viewed, device, traffic source, user type, and conversion flag.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 12000
dates = pd.date_range('2024-01-01', periods=60, freq='D')
session_dates = np.random.choice(dates, size=n)
hour_w = np.array([1,1,1,1,1,2,3,5,7,8,8,7,6,7,7,6,5,5,4,4,3,3,2,1])
hours = np.random.choice(range(24), size=n, p=hour_w/hour_w.sum())

device = np.random.choice(['desktop', 'mobile', 'tablet'], size=n, p=[0.42, 0.48, 0.10])
source = np.random.choice(['organic', 'direct', 'social', 'paid', 'email'], size=n,
                           p=[0.35, 0.25, 0.20, 0.12, 0.08])
user_type = np.random.choice(['new', 'returning'], size=n, p=[0.55, 0.45])

# Duration model: lognormal base + device/source modifiers
device_mult = {'desktop': 1.3, 'mobile': 0.75, 'tablet': 1.0}
source_mult = {'organic': 1.0, 'direct': 1.2, 'social': 0.6, 'paid': 0.8, 'email': 1.4}
user_mult = {'new': 0.8, 'returning': 1.3}

duration = []
for i in range(n):
    base = np.random.lognormal(4.5, 1.2)  # median ~90s
    d = base * device_mult[device[i]] * source_mult[source[i]] * user_mult[user_type[i]]
    d = max(1, min(d, 3600))  # cap at 1 hour
    duration.append(round(d, 1))

duration = np.array(duration)
# Pages: correlated with duration
pages = np.clip(np.round(duration / 45 + np.random.normal(0, 0.8, n)), 1, 50).astype(int)
# Bounce: sessions < 10 seconds
bounce = (duration < 10).astype(int)
# Conversion: higher for longer sessions
conv_prob = np.clip(0.005 + 0.0003 * duration, 0, 0.15)
converted = np.random.binomial(1, conv_prob)

df = pd.DataFrame({
    'date': session_dates, 'hour': hours, 'device': device,
    'source': source, 'user_type': user_type,
    'duration_sec': duration, 'pages_viewed': pages,
    'bounce': bounce, 'converted': converted
})
df['duration_min'] = (df['duration_sec'] / 60).round(2)
df['day_of_week'] = pd.to_datetime(df['date']).dt.day_name()

print(f'Dataset: {df.shape}')
print(f'Median session: {df["duration_sec"].median():.0f}s ({df["duration_min"].median():.1f} min)')
print(f'Bounce rate: {df["bounce"].mean()*100:.1f}%')
print(f'Conversion rate: {df["converted"].mean()*100:.2f}%')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nDevice distribution:\n{df["device"].value_counts()}')
print(f'\nSource distribution:\n{df["source"].value_counts()}')
print(f'\nDuration stats (seconds):')
print(df['duration_sec'].describe().round(1))

## Session Duration Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Raw distribution
axes[0].hist(df['duration_sec'], bins=80, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_title('Session Duration Distribution')
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['duration_sec'].median(), color='red', linestyle='--', label=f'Median: {df["duration_sec"].median():.0f}s')
axes[0].legend()

# Log scale
axes[1].hist(np.log10(df['duration_sec'] + 1), bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[1].set_title('Session Duration Distribution (log10)')
axes[1].set_xlabel('log10(Duration seconds)')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'duration_distribution.png'), dpi=100, bbox_inches='tight')
plt.show()

## Engagement Tiers

In [ ]:
bins = [0, 10, 60, 300, 3601]
labels = ['Bounce (<10s)', 'Casual (10-60s)', 'Engaged (1-5min)', 'Power (5min+)']
df['tier'] = pd.cut(df['duration_sec'], bins=bins, labels=labels)
tier_stats = df.groupby('tier', observed=True).agg(
    sessions=('duration_sec', 'count'),
    avg_pages=('pages_viewed', 'mean'),
    conversion_rate=('converted', 'mean')
).round(3)
tier_stats['pct'] = (tier_stats['sessions'] / len(df) * 100).round(1)
print(tier_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
tier_stats['pct'].plot.bar(ax=axes[0], color=['#F44336','#FF9800','#4CAF50','#2196F3'], edgecolor='black')
axes[0].set_title('Session Tier Distribution')
axes[0].set_ylabel('% of Sessions')
axes[0].tick_params(axis='x', rotation=30)

tier_stats['conversion_rate'].plot.bar(ax=axes[1], color=['#F44336','#FF9800','#4CAF50','#2196F3'], edgecolor='black')
axes[1].set_title('Conversion Rate by Engagement Tier')
axes[1].set_ylabel('Conversion Rate')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'engagement_tiers.png'), dpi=100, bbox_inches='tight')
plt.show()

## Device Effects

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='device', y='duration_sec', ax=ax, showfliers=False)
ax.set_title('Session Duration by Device')
ax.set_ylabel('Duration (seconds)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'device_duration.png'), dpi=100, bbox_inches='tight')
plt.show()

print('Median duration by device:')
print(df.groupby('device')['duration_sec'].median().round(0))
print('\nBounce rate by device:')
print((df.groupby('device')['bounce'].mean() * 100).round(1))

## Traffic Source Effects

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
source_order = df.groupby('source')['duration_sec'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='source', y='duration_sec', order=source_order, ax=ax, showfliers=False)
ax.set_title('Session Duration by Traffic Source')
ax.set_ylabel('Duration (seconds)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'source_duration.png'), dpi=100, bbox_inches='tight')
plt.show()

print('Median duration and bounce rate by source:')
src_stats = df.groupby('source').agg(
    median_sec=('duration_sec', 'median'),
    bounce_rate=('bounce', 'mean'),
    conversion=('converted', 'mean')
).round(3)
print(src_stats.sort_values('median_sec', ascending=False))

## Time-of-Day Pattern

In [ ]:
hourly = df.groupby('hour').agg(
    sessions=('duration_sec', 'count'),
    median_duration=('duration_sec', 'median'),
    bounce_rate=('bounce', 'mean')
)
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(hourly.index, hourly['sessions'], alpha=0.3, color='steelblue', label='Sessions')
ax1.set_ylabel('Session Count', color='steelblue')
ax2 = ax1.twinx()
ax2.plot(hourly.index, hourly['median_duration'], color='red', marker='o', label='Median Duration')
ax2.set_ylabel('Median Duration (s)', color='red')
ax1.set_xlabel('Hour of Day')
ax1.set_title('Sessions and Duration by Hour')
ax1.set_xticks(range(24))
fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.95))
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'hourly_pattern.png'), dpi=100, bbox_inches='tight')
plt.show()

## New vs Returning Users

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='user_type', y='duration_sec', ax=ax, showfliers=False)
ax.set_title('Session Duration: New vs Returning Users')
ax.set_ylabel('Duration (seconds)')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'user_type_duration.png'), dpi=100, bbox_inches='tight')
plt.show()

print('Stats by user type:')
print(df.groupby('user_type').agg(
    median_sec=('duration_sec', 'median'),
    bounce_rate=('bounce', 'mean'),
    conversion=('converted', 'mean'),
    avg_pages=('pages_viewed', 'mean')
).round(3))

## Interpretation and Business Insight
- **Session duration is highly right-skewed** — median ~90s but mean is much higher due to power users
- **Bounce sessions** (<10s) are ~15% of traffic — focus on landing page optimization for these
- **Engaged and power users** convert at 5-10× the rate of casual visitors
- **Desktop sessions** are 30-40% longer than mobile — mobile UX needs improvement
- **Email traffic** produces the longest sessions — email subscribers are the most engaged audience
- **Social traffic** has the shortest sessions and highest bounce — social ads attract low-intent visitors
- **Returning users** have 50%+ longer sessions — familiarity drives deeper engagement
- **Evening sessions** (7-9pm) tend to be slightly longer — users have more time to browse

## Limitations
- Synthetic data — real session data has complex tab-switching, idle detection, and bot traffic
- No content/page-level analysis (which pages cause bounces)
- No scroll depth or interaction events
- Session definition is simplified (no timeout-based splitting)
- Conversion is modeled as duration-correlated, which is circular

## How to Improve This Project
- Add page-level engagement data (scroll depth, clicks, video plays)
- Include content type analysis (blog vs product vs checkout)
- Build session quality scoring model
- Add funnel analysis within sessions
- Include bot detection and filtering

## Production Considerations
- Real-time session monitoring dashboards
- Automated alerting for bounce rate spikes
- Session replay integration for UX research
- Personalized content based on predicted session intent
- A/B testing of layout changes on session duration

## Common Mistakes
- Using mean session duration instead of median (skewness distorts the mean)
- Counting bounce sessions in average duration (artificially lowers it)
- Not distinguishing device types when reporting engagement
- Treating all short sessions as failures (some tasks are legitimately quick)
- Ignoring the contribution of power users to total engagement time

## Mini Challenge / Exercises
1. What percentage of total engagement time comes from the "Power" tier?
2. Calculate the 90th percentile session duration by device — how different are they?
3. If you could eliminate 50% of bounces, what would the new overall conversion rate be?
4. Build a simple decision tree to predict engagement tier from device, source, and user type.

## Final Summary / Key Takeaways
- Session duration follows a lognormal distribution — always report median, not mean
- Engagement tiers reveal distinct user segments with very different conversion rates
- Device and traffic source are the strongest predictors of session quality
- Returning users are significantly more engaged — retention drives engagement
- Reducing bounces and lengthening casual sessions offers the highest conversion ROI